<a href="https://colab.research.google.com/github/yosungcho/yosungcho.github.io/blob/main/11132025InferenceModelNoMQTT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
from PIL import Image
import os
import numpy as np
import pandas as pd

# --- 1. SET YOUR MODEL PATHS HERE ---
model_paths = {
    'DR': '/content/drive/MyDrive/Jojo CNN/Diabetic/Trained Model/DiabeticRunModela.h5',
    'Cataract': '/content/drive/MyDrive/Jojo CNN/cataract/Cataract Trained Model/CataractRunModela.h5',
    'Glaucoma': '/content/drive/MyDrive/Jojo CNN/Glaucoma/Glaucoma Trained Model/GlaucomaRunModela.h5'
}

# --- 2. LOAD ALL MODELS ---
print("Loading models...")
models = {}
for name, path in model_paths.items():
    try:
        models[name] = tf.keras.models.load_model(path)
        print(f"Successfully loaded {name} from {path}")
    except Exception as e:
        print(f"Error loading {name} from {path}. Skipping. Error: {e}")
print("--- All models loaded ---")


# --- 3. HELPER FUNCTION FOR YOUR LOGIC ---
# This function applies your custom rules and updates the stats
def process_prediction(prediction_value):
    """
    Returns the predicted class based on the prediction value.
    """
    if prediction_value >= 0.8:
        predicted_class = 'Malignant'
    elif prediction_value <= 0.2:
        predicted_class = 'Benign'
    else:
        predicted_class = 'Uncertain'
    return predicted_class


# --- 4. SETUP IMAGE FOLDER AND STATS COUNTERS AND RESULTS LIST ---
image_folder_path = '/content/drive/MyDrive/Jojo CNN/NewTestingImagesAA'
image_files = [f for f in os.listdir(image_folder_path) if f.endswith('.jpg')]

# Create a dictionary to hold the stats for EACH model
model_stats = {name: {'Malignant': 0, 'Benign': 0, 'Uncertain': 0} for name in models.keys()}

# List to store results for the table
image_results_list = []

# --- 5. PROCESS ALL IMAGES WITH CORRECTED PREPROCESSING ---
print(f"\nProcessing {len(image_files)} images...")
print("✅ Preprocessing matches training: RGB conversion + 128x128 resize + [0,1] normalization\n")

for image_file in image_files:
    image_path = os.path.join(image_folder_path, image_file)

    # Preprocess the image ONCE - MATCH TRAINING EXACTLY
    try:
        test_image = Image.open(image_path)
        test_image = test_image.convert('RGB')  # ✅ CRITICAL FIX - Match training!
        test_image = test_image.resize((128, 128))
        test_image_np = np.array(test_image) / 255.0
        test_image_batch = np.expand_dims(test_image_np, axis=0)  # Create a batch of 1

        # Verify shape matches training
        expected_shape = (1, 128, 128, 3)
        if test_image_batch.shape != expected_shape:
            print(f"⚠️  Shape mismatch for {image_file}: {test_image_batch.shape} (expected {expected_shape})")
            continue

    except Exception as e:
        print(f"Error processing image {image_file}. Skipping. Error: {e}")
        continue  # Skip to the next image if there's an error

    print(f"\n--- Results for: {image_file} ---")

    image_result = {'Image': image_file}

    # Loop through your loaded models and get a prediction from each
    for model_name, model in models.items():
        # Make prediction
        try:
            prediction_value = model.predict(test_image_batch, verbose=0)[0][0]
        except Exception as e:
            print(f"Error predicting with {model_name} for image {image_file}. Error: {e}")
            prediction_value = None  # Indicate prediction failed

        if prediction_value is not None:
            # Apply your logic and update counters using the helper function
            predicted_class = process_prediction(prediction_value)
            model_stats[model_name][predicted_class] += 1
            image_result[model_name] = f"{predicted_class} ({prediction_value:.4f})"
            # Print the result for this specific model
            print(f"  {model_name}: {predicted_class} (Raw: {prediction_value:.4f})")
        else:
            image_result[model_name] = "Prediction Failed"
            print(f"  {model_name}: Prediction Failed")

    image_results_list.append(image_result)


# --- 6. PRINT FINAL SUMMARY ---
print("\n" + "="*30)
print("--- FINAL TALLY ---")
print("="*30)

total_malignant = 0
total_benign = 0
total_uncertain = 0

for model_name, stats in model_stats.items():
    print(f"\nResults for {model_name}:")
    print(f"  Total Malignant:  {stats['Malignant']}")
    print(f"  Total Benign:     {stats['Benign']}")
    print(f"  Total Uncertain: {stats['Uncertain']}")
    total_images_processed_by_model = stats['Malignant'] + stats['Benign'] + stats['Uncertain']
    print(f"  (Total Images Processed by {model_name}: {total_images_processed_by_model})")

    total_malignant += stats['Malignant']
    total_benign += stats['Benign']
    total_uncertain += stats['Uncertain']

print("\n" + "="*30)
print("--- OVERALL TALLY ---")
print("="*30)
print(f"Total Malignant across all models: {total_malignant}")
print(f"Total Benign across all models:    {total_benign}")
print(f"Total Uncertain across all models: {total_uncertain}")
print(f"(Total Images Processed: {len(image_files)})")


# --- 7. DISPLAY RESULTS TABLE ---
print("\n" + "="*30)
print("--- IMAGE-WISE RESULTS TABLE ---")
print("="*30)

results_df = pd.DataFrame(image_results_list)
# Reorder columns
desired_column_order = ['Image', 'Cataract', 'DR', 'Glaucoma']
# Ensure all desired columns are in the dataframe, add missing ones if necessary
desired_column_order = [col for col in desired_column_order if col in results_df.columns]
results_df = results_df[desired_column_order]

display(results_df)

print("\n✅ Processing complete with corrected preprocessing!")

Loading models...


Successfully loaded DR from /content/drive/MyDrive/Jojo CNN/Diabetic/Trained Model/DiabeticRunModela.h5


Successfully loaded Cataract from /content/drive/MyDrive/Jojo CNN/cataract/Cataract Trained Model/CataractRunModela.h5


Successfully loaded Glaucoma from /content/drive/MyDrive/Jojo CNN/Glaucoma/Glaucoma Trained Model/GlaucomaRunModela.h5
--- All models loaded ---

Processing 30 images...
✅ Preprocessing matches training: RGB conversion + 128x128 resize + [0,1] normalization


--- Results for: CA1.jpg ---
  DR: Malignant (Raw: 0.9693)
  Cataract: Malignant (Raw: 0.8983)
  Glaucoma: Malignant (Raw: 0.9563)

--- Results for: DRN2.jpg ---
  DR: Benign (Raw: 0.0014)
  Cataract: Benign (Raw: 0.0006)
  Glaucoma: Benign (Raw: 0.0040)

--- Results for: CA2.jpg ---
  DR: Malignant (Raw: 0.9840)
  Cataract: Benign (Raw: 0.0067)
  Glaucoma: Benign (Raw: 0.0161)

--- Results for: GLA1.jpg ---
  DR: Uncertain (Raw: 0.7859)
  Cataract: Malignant (Raw: 0.9992)
  Glaucoma: Malignant (Raw: 1.0000)

--- Results for: CA5.jpg ---
  DR: Malignant (Raw: 0.9726)
  Cataract: Malignant (Raw: 0.9745)
  Glaucoma: Malignant (Raw: 0.8230)

--- Results for: DR2.jpg ---
  DR: Malignant (Raw: 1.0000)
  Cataract: Malignant (Raw: 0.9669

,Image,Cataract,DR,Glaucoma
0,CA1.jpg,Malignant (0.8983),Malignant (0.9693),Malignant (0.9563)
1,DRN2.jpg,Benign (0.0006),Benign (0.0014),Benign (0.0040)
2,CA2.jpg,Benign (0.0067),Malignant (0.9840),Benign (0.0161)
3,GLA1.jpg,Malignant (0.9992),Uncertain (0.7859),Malignant (1.0000)
4,CA5.jpg,Malignant (0.9745),Malignant (0.9726),Malignant (0.8230)
5,DR2.jpg,Malignant (0.9669),Malignant (1.0000),Malignant (0.8836)
6,CA3.jpg,Uncertain (0.6627),Malignant (1.0000),Malignant (0.9283)
7,DR4.jpg,Benign (0.1245),Malignant (0.9993),Uncertain (0.7732)
8,DR3.jpg,Uncertain (0.7632),Malignant (0.9552),Malignant (0.8948)
9,DRN5.jpg,Benign (0.0501),Benign (0.0005),Benign (0.0059)



✅ Processing complete with corrected preprocessing!
